# 16 · Data lake: silver GeoParquet tables (Part D, step D1)

**What this does:** loads every final layer into one data lake with Apache Spark + Sedona and stores it as
**GeoParquet** (columnar, compressed, with a geometry column), the format all later Part D steps read.

| Lake tier | What it holds | Where |
|---|---|---|
| **Bronze** | the raw and final files as collected (CSV, GeoJSON, GeoPackage, GeoTIFF) — not copied again | `data/raw/`, `FINAL COMPLETE DATASET/` |
| **Silver** | cleaned, typed tables with geometry, one folder per table (this notebook) | `data/lake/silver/` |
| **Gold** | results: access per cell and per area, inequality indices (step D6) | `data/lake/gold/` |

**Silver tables:** `health_facilities` (India), `schools_india` (partitioned by state, with national level flags
primary 1–5, upper primary 6–8, secondary 9–10, higher secondary 11–12 and students per level), `schools_kerala`
(Sametham levels LP 1–4, UP 5–7, HS 8–10, HSS and 2026-27 students per level), `fire_stations` (Kerala),
`states`, `districts`, `subdistricts` (geoBoundaries) and `kerala_local_bodies` (OSM, admin_level 8).
Rasters (population) are added in D2 and the Kerala roads in D5.

**Nothing in the source files is changed**; they are only read. Coordinates are WGS84 longitude/latitude.

**Checks:** each table is read back from disk; rows, geometries and key column sums are compared with an
independent pandas read of the same file; every geometry must be valid; a Sedona spatial join checks that the
Kerala schools fall inside Kerala. A catalog of the tables is written to `data/lake/silver/_catalog.csv`.

## Project folder (laptop or Colab)

In [1]:
# Project folder. Laptop: open the notebook from "PROJECTS IPYNB" (or the project root).
# Google Colab: upload the project folder to Google Drive ("My Drive") and set its name below.
from pathlib import Path
import sys
PROJECT_FOLDER = "Distance-Capacity-and-Inequality-in-Access-to-Government-and-Aided-Schools"  # Colab only: folder name inside My Drive
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive") / PROJECT_FOLDER
else:
    PROJECT = Path.cwd().parent if Path.cwd().name in ("notebooks", "PROJECTS IPYNB") else Path.cwd()
assert (PROJECT / "scripts").is_dir(), f"project folder not found: {PROJECT}"
sys.path.insert(0, str(PROJECT / "scripts"))

In [2]:
# Colab only: install packages that Colab does not have by default (does nothing on the laptop)
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rapidfuzz", "pymupdf", "pyogrio", "rasterio"], check=True)

## Install the Part D packages

In [3]:
# Part D packages, pinned in requirements-bigdata.txt (PySpark 3.5.9, apache-sedona 1.9.1, pyarrow).
# Colab also needs Java 17, which Spark runs on (the laptop already has it: JAVA_HOME).
import os
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT / "requirements-bigdata.txt")], check=True)
if IN_COLAB:
    subprocess.run("apt-get install -y -qq openjdk-17-jdk-headless > /dev/null", shell=True, check=True)
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

## Start Spark + Sedona

In [4]:
import geopandas as gpd
import pandas as pd
import pyogrio
from pyspark.sql import functions as F
from pyspark.sql.types import BinaryType, StringType, StructField, StructType
from bigdata.spark_session import get_spark

spark = get_spark(PROJECT)
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
FINAL = PROJECT / "FINAL COMPLETE DATASET"
RAW = PROJECT / "data" / "raw"
SILVER = PROJECT / "data" / "lake" / "silver"
SILVER.mkdir(parents=True, exist_ok=True)
print("cores:", spark.sparkContext.defaultParallelism, "| Spark UI:", spark.sparkContext.uiWebUrl)
print("silver tables go to:", SILVER)

cores: 12 | Spark UI: http://127.0.0.1:4040
silver tables go to: C:\Users\ASUS\Desktop\sem3\BIG SPATIAL DATA\PROJECT- SPATIAL INEQUALITY ASESSMENT\data\lake\silver


## Helpers: read, add geometry, write and check

In [5]:
def read_csv(path, numeric=(), boolean=()):
    """Spark CSV reader. Every column is read as text first (keeps codes such as UDISE 0101... with their leading zero),
    then only the listed columns are cast. multiLine + escape follow the quoting pandas used when it wrote the files."""
    df = spark.read.csv(str(path), header=True, multiLine=True, escape='"')
    for c in numeric:
        df = df.withColumn(c, F.col(c).cast("double"))
    for c in boolean:
        df = df.withColumn(c, F.col(c).cast("boolean"))
    return df


def with_points(df):
    """Point geometry from lon/lat (WGS84). Rows without coordinates keep a null geometry; they stay in the table."""
    return df.withColumn("geometry", F.expr("CASE WHEN lat IS NOT NULL AND lon IS NOT NULL THEN ST_Point(lon, lat) END"))


def read_polygons(path, where=None):
    """Boundary file -> Spark: GeoPandas reads it, geometries go to Spark as WKB, Sedona turns them back into geometry."""
    g = gpd.read_file(path)
    if where:
        g = g.query(where)
    attrs = g.drop(columns="geometry").astype("string").astype(object)
    pdf = attrs.where(attrs.notna(), None).assign(wkb=g.geometry.to_wkb())
    schema = StructType([StructField(c, StringType()) for c in attrs.columns] + [StructField("wkb", BinaryType())])
    return spark.createDataFrame(pdf, schema).withColumn("geometry", F.expr("ST_GeomFromWKB(wkb)")).drop("wkb")


catalog = []


def write_silver(df, name, source, partition=None):
    """Write one GeoParquet table, read it back from disk and check rows and geometries."""
    out = SILVER / name
    w = (df.repartition(partition) if partition else df.coalesce(1)).write.mode("overwrite").format("geoparquet")
    (w.partitionBy(partition) if partition else w).save(str(out))
    back = spark.read.format("geoparquet").load(str(out))
    stats = back.selectExpr("count(*) AS rows", "count(geometry) AS with_geometry",
                            "sum(CASE WHEN geometry IS NOT NULL AND NOT ST_IsValid(geometry) THEN 1 ELSE 0 END) AS invalid").first()
    files = [f for f in out.rglob("*.parquet")]
    row = {"table": name, "rows": stats["rows"], "with_geometry": stats["with_geometry"], "invalid_geometry": stats["invalid"],
           "files": len(files), "size_mb": round(sum(f.stat().st_size for f in files) / 1e6, 1),
           "partitioned_by": partition or "", "source": source}
    catalog.append(row)
    print(row)
    assert stats["invalid"] == 0, f"{name}: invalid geometries"
    return back


def same_as_pandas(back, path, cols):
    """Independent check of the Spark CSV parse: row count, non-null count and sum of key columns equal pandas."""
    p = pd.read_csv(path, usecols=cols, low_memory=False)
    s = back.select(*[F.col(c).cast("double") for c in cols]).toPandas()
    assert len(p) == len(s), f"rows differ: pandas {len(p)}, spark {len(s)}"
    for c in cols:
        assert p[c].notna().sum() == s[c].notna().sum(), f"{c}: non-null counts differ"
        assert abs(p[c].sum() - s[c].sum()) <= 1e-6 * max(1.0, abs(p[c].sum())), f"{c}: sums differ"
    print("same as pandas:", len(p), "rows; columns checked:", ", ".join(cols))

## Health facilities (India)

In [6]:
src = FINAL / "health" / "india_health_facilities_final.csv"
health = with_points(read_csv(src, numeric=["lat", "lon", "nin_vs_hcd_m", "km_outside_state"], boolean=["in_access_analysis"]))
health_t = write_silver(health, "health_facilities", "FINAL COMPLETE DATASET/health/india_health_facilities_final.csv")
same_as_pandas(health_t, src, ["lat", "lon"])
health_t.groupBy("std_type").agg(F.count("*").alias("facilities"), F.count("geometry").alias("with_location"),
                                  F.sum(F.col("in_access_analysis").cast("int")).alias("in_access_analysis")) \
        .orderBy(F.desc("facilities")).show(20)

{'table': 'health_facilities', 'rows': 228986, 'with_geometry': 228230, 'invalid_geometry': 0, 'files': 1, 'size_mb': 23.7, 'partitioned_by': '', 'source': 'FINAL COMPLETE DATASET/health/india_health_facilities_final.csv'}


same as pandas: 228986 rows; columns checked: lat, lon


+--------+----------+-------------+------------------+
|std_type|facilities|with_location|in_access_analysis|
+--------+----------+-------------+------------------+
|      SC|    181575|       180889|            180889|
|     PHC|     32737|        32687|             32687|
|     CHC|      6381|         6365|              6365|
|     OTH|      2173|         2173|              2173|
|     UHC|      1418|         1418|              1418|
|     SDH|      1401|         1399|              1399|
|      DH|      1132|         1132|              1132|
|    DISP|       752|          750|               733|
|     UHP|       436|          436|               436|
|    HOSP|       414|          414|               414|
|     MAT|       374|          374|               374|
|     MCH|       146|          146|               146|
|    SPEC|        29|           29|                14|
|     MOB|        18|           18|                 0|
+--------+----------+-------------+------------------+



## Schools, India (national levels, partitioned by state)

In [7]:
# National school levels (India run): a school counts for every level whose classes it teaches (class_from..class_to);
# capacity of a level = UDISE students enrolled in that level's classes.
ROMAN = ["i", "ii", "iii", "iv", "v", "vi", "vii", "viii", "ix", "x", "xi", "xii"]
NATIONAL_LEVELS = {"primary": (1, 5), "upper_primary": (6, 8), "secondary": (9, 10), "higher_secondary": (11, 12)}

src = FINAL / "schools" / "india_schools_govt_aided.csv"
student_cols = [f"{r}_students" for r in ROMAN]
numeric = ["class_from", "class_to", "lat", "lon", "class_rooms", "total_teachers", "pre_primary_students", *student_cols,
           "class_students", "km_outside_state", "udise_lat", "udise_lon", "osm_score", "moved_m"]
schools = read_csv(src, numeric=numeric, boolean=["placeholder", "low_precision", "outside_state", "village_outlier"])
for level, (a, b) in NATIONAL_LEVELS.items():
    schools = schools.withColumn(f"has_{level}", (F.col("class_from") <= b) & (F.col("class_to") >= a)) \
                     .withColumn(f"cap_{level}", sum(F.coalesce(F.col(f"{ROMAN[k - 1]}_students"), F.lit(0.0)) for k in range(a, b + 1)))
schools_t = write_silver(with_points(schools), "schools_india", "FINAL COMPLETE DATASET/schools/india_schools_govt_aided.csv",
                         partition="state_name")
same_as_pandas(schools_t, src, ["lat", "lon", "class_students", "i_students", "xii_students"])

{'table': 'schools_india', 'rows': 1070017, 'with_geometry': 1070016, 'invalid_geometry': 0, 'files': 35, 'size_mb': 125.4, 'partitioned_by': 'state_name', 'source': 'FINAL COMPLETE DATASET/schools/india_schools_govt_aided.csv'}


same as pandas: 1070017 rows; columns checked: lat, lon, class_students, i_students, xii_students


### Level counts and checks

In [8]:
# Level counts, and two consistency checks of the level columns
levels = list(NATIONAL_LEVELS)
schools_t.select([F.sum(F.col(f"has_{l}").cast("int")).alias(f"schools_{l}") for l in levels] +
                 [F.sum(f"cap_{l}").cast("long").alias(f"students_{l}") for l in levels]).show(vertical=True)
c = schools_t.selectExpr("sum(CASE WHEN abs(" + " + ".join(f"cap_{l}" for l in levels) + " - class_students) > 0.5 THEN 1 ELSE 0 END) AS sum_differs",
                         "sum(CASE WHEN class_students IS NULL THEN 1 ELSE 0 END) AS no_class_students",
                         "sum(CASE WHEN class_from IS NULL OR class_to IS NULL THEN 1 ELSE 0 END) AS no_class_range").first()
# students recorded in a level the school's class range does not include (a UDISE inconsistency, reported, not changed)
outside = schools_t.select([F.sum(((F.col(f"cap_{l}") > 0) & ~F.col(f"has_{l}")).cast("int")).alias(l) for l in levels]).first()
print("level capacities that do not add up to class_students:", c["sum_differs"], "| no class_students:", c["no_class_students"],
      "| no class range:", c["no_class_range"])
print("schools with students in a level outside their class range:", outside.asDict())
schools_t.groupBy("state_name").count().orderBy(F.desc("count")).show(40, truncate=False)

-RECORD 0-----------------------------
 schools_primary           | 911383   
 schools_upper_primary     | 407441   
 schools_secondary         | 145333   
 schools_higher_secondary  | 68422    
 students_primary          | 71754450 
 students_upper_primary    | 41228352 
 students_secondary        | 23641440 
 students_higher_secondary | 14800014 



level capacities that do not add up to class_students: 0 | no class_students: 392 | no class range: 392
schools with students in a level outside their class range: {'primary': 1, 'upper_primary': 0, 'secondary': 0, 'higher_secondary': 0}


+---------------------------+------+
|state_name                 |count |
+---------------------------+------+
|Uttar Pradesh              |149506|
|Madhya Pradesh             |92696 |
|Maharashtra                |86438 |
|West Bengal                |75259 |
|Bihar                      |73750 |
|Rajasthan                  |69279 |
|Odisha                     |53390 |
|Karnataka                  |53349 |
|Chhattisgarh               |48312 |
|Assam                      |46768 |
|Tamil Nadu                 |45848 |
|Andhra Pradesh             |43891 |
|Gujarat                    |39342 |
|Jharkhand                  |36513 |
|Telangana                  |30575 |
|Jammu and Kashmir          |23038 |
|Uttarakhand                |16792 |
|Haryana                    |14138 |
|Punjab                     |14106 |
|Himachal Pradesh           |13678 |
|Kerala                     |11768 |
|Meghalaya                  |11623 |
|Tripura                    |4310  |
|Manipur                    |3466  |
|

## Schools, Kerala (Sametham levels)

In [9]:
# Kerala case study: Sametham levels LP 1-4, UP 5-7, HS 8-10, HSS 11-12. Level flags come from the verified layer
# (has_lp ... has_hss); capacity = 2026-27 Sametham students in that level's classes. Both source files are only read.
KERALA_LEVELS = {"lp": [f"sam_students_std{k:02d}" for k in range(1, 5)], "up": [f"sam_students_std{k:02d}" for k in range(5, 8)],
                 "hs": [f"sam_students_std{k:02d}" for k in range(8, 11)], "hss": ["sam_students_hss_total"]}
src = FINAL / "schools" / "kerala_schools_capacity.csv"
std_cols = [c for cols in KERALA_LEVELS.values() for c in cols]
cap = read_csv(src, numeric=["lat", "lon", *std_cols, "sam_students_total", "sam_students_hss_plus1", "sam_students_hss_plus2",
                             "capacity_students", "classrooms", "teachers",
                             "teachers_lp", "teachers_up", "teachers_hs", "teachers_hss",  # class teachers per level (notebook 09)
                             "hss_batches", "hss_batches_unaided", "hss_seats_per_year", "hss_capacity_2yr",  # HSCAP 2026
                             "vhse_batches", "vhse_seats_per_year", "vhse_capacity_2yr"],  # VHSCAP 2026
               boolean=["sam_is_hilly", "sam_is_coastal"])
layer = read_csv(FINAL / "schools" / "kerala_schools_govt_aided_final.csv", boolean=["has_lp", "has_up", "has_hs", "has_hss"]) \
        .select("school_code_num", "has_lp", "has_up", "has_hs", "has_hss", "block", "local_body", "location_source", "location_accuracy")
kerala = cap.join(layer, "school_code_num", "inner")
for level, cols in KERALA_LEVELS.items():
    kerala = kerala.withColumn(f"cap_{level}", sum(F.coalesce(F.col(c), F.lit(0.0)) for c in cols))
kerala_t = write_silver(with_points(kerala), "schools_kerala", "FINAL COMPLETE DATASET/schools/kerala_schools_capacity.csv + "
                        "kerala_schools_govt_aided_final.csv (level flags)")
assert kerala_t.count() == 11764, "every school with capacity must find its row in the verified layer"
same_as_pandas(kerala_t, src, ["lat", "lon", "sam_students_std01", "sam_students_std10", "sam_students_hss_total"])
kerala_t.select([F.sum(F.col(f"has_{l}").cast("int")).alias(f"schools_{l}") for l in KERALA_LEVELS] +
                [F.sum(f"cap_{l}").cast("long").alias(f"students_{l}") for l in KERALA_LEVELS]).show(vertical=True)
std1_10 = kerala_t.groupBy("finance_type").agg(F.sum(F.col("cap_lp") + F.col("cap_up") + F.col("cap_hs")).cast("long").alias("std1_10")).toPandas()
print(std1_10.to_string(index=False), "| state table 2026-27: Government 1,033,009, Aided 1,997,616")

{'table': 'schools_kerala', 'rows': 11764, 'with_geometry': 11763, 'invalid_geometry': 0, 'files': 1, 'size_mb': 2.9, 'partitioned_by': '', 'source': 'FINAL COMPLETE DATASET/schools/kerala_schools_capacity.csv + kerala_schools_govt_aided_final.csv (level flags)'}


same as pandas: 11764 rows; columns checked: lat, lon, sam_students_std01, sam_students_std10, sam_students_hss_total
-RECORD 0---------------
 schools_lp   | 9282    
 schools_up   | 4778    
 schools_hs   | 2655    
 schools_hss  | 1865    
 students_lp  | 960557  
 students_up  | 928384  
 students_hs  | 1141074 
 students_hss | 727678  



finance_type  std1_10
  Government  1032865
       Aided  1997150 | state table 2026-27: Government 1,033,009, Aided 1,997,616


## Fire stations (Kerala)

In [10]:
src = FINAL / "fire_stations" / "kerala_fire_stations.csv"
fire = with_points(read_csv(src, numeric=["sl_no", "lat", "lon", "osm_vs_official_m", "manual_vs_osm_m", "manual_vs_official_m",
                                          "nearest_station_m"]))
fire_t = write_silver(fire, "fire_stations", "FINAL COMPLETE DATASET/fire_stations/kerala_fire_stations.csv")
same_as_pandas(fire_t, src, ["lat", "lon"])

{'table': 'fire_stations', 'rows': 129, 'with_geometry': 129, 'invalid_geometry': 0, 'files': 1, 'size_mb': 0.0, 'partitioned_by': '', 'source': 'FINAL COMPLETE DATASET/fire_stations/kerala_fire_stations.csv'}
same as pandas: 129 rows; columns checked: lat, lon


## Boundaries: states, districts, sub-districts, Kerala local bodies

In [11]:
B = RAW / "boundaries"
for name, f in [("states", "geoBoundaries-IND-ADM1.geojson"), ("districts", "geoBoundaries-IND-ADM2.geojson"),
                ("subdistricts", "geoBoundaries-IND-ADM3.geojson")]:
    t = write_silver(read_polygons(B / f), name, f"data/raw/boundaries/{f} (geoBoundaries)")
    assert t.count() == pyogrio.read_info(B / f)["features"], name  # feature count in the source file

# Kerala local bodies: OSM boundary=local_authority, admin_level 8 = grama panchayats, municipalities and corporations
f = RAW / "osm" / "kerala_local_bodies_2026-11-07.gpkg"
lb = write_silver(read_polygons(f, where="admin_level == '8'"), "kerala_local_bodies", "data/raw/osm/kerala_local_bodies_2026-11-07.gpkg (OSM, admin_level 8)")
lb.groupBy("local_authority_IN").count().show()
assert lb.count() == 1027

{'table': 'states', 'rows': 36, 'with_geometry': 36, 'invalid_geometry': 0, 'files': 1, 'size_mb': 16.9, 'partitioned_by': '', 'source': 'data/raw/boundaries/geoBoundaries-IND-ADM1.geojson (geoBoundaries)'}


{'table': 'districts', 'rows': 735, 'with_geometry': 735, 'invalid_geometry': 0, 'files': 1, 'size_mb': 29.6, 'partitioned_by': '', 'source': 'data/raw/boundaries/geoBoundaries-IND-ADM2.geojson (geoBoundaries)'}


{'table': 'subdistricts', 'rows': 6824, 'with_geometry': 6824, 'invalid_geometry': 0, 'files': 1, 'size_mb': 38.5, 'partitioned_by': '', 'source': 'data/raw/boundaries/geoBoundaries-IND-ADM3.geojson (geoBoundaries)'}


{'table': 'kerala_local_bodies', 'rows': 1027, 'with_geometry': 1027, 'invalid_geometry': 0, 'files': 1, 'size_mb': 3.3, 'partitioned_by': '', 'source': 'data/raw/osm/kerala_local_bodies_2026-11-07.gpkg (OSM, admin_level 8)'}
+--------------------+-----+
|  local_authority_IN|count|
+--------------------+-----+
|municipal_corpora...|    6|
|        municipality|   85|
|      gram_panchayat|  936|
+--------------------+-----+



## Spatial check on the lake

In [12]:
# Spatial read-back test on the lake (Sedona): every Kerala school point must fall inside the Kerala state polygon
spark.read.format("geoparquet").load(str(SILVER / "states")).createOrReplaceTempView("states")
spark.read.format("geoparquet").load(str(SILVER / "schools_kerala")).createOrReplaceTempView("schools_kerala")
inside = spark.sql("""SELECT s.shapeName AS state, count(*) AS schools FROM schools_kerala k
                      JOIN states s ON ST_Intersects(s.geometry, k.geometry) GROUP BY s.shapeName""").toPandas()
print(inside.to_string(index=False))
print("Kerala schools in the lake:", kerala_t.count(), "| matched to a state polygon:", int(inside["schools"].sum()))

     state  schools
    Kerala    11741
 Karnātaka        3
Puducherry        8
Kerala schools in the lake: 11764 | matched to a state polygon: 11752


## Catalog

In [13]:
cat = pd.DataFrame(catalog)
old = SILVER / "_catalog.csv"
if old.exists():  # keep rows of tables written by later notebooks (e.g. population_cells, notebook 17)
    prev = pd.read_csv(old)
    cat = pd.concat([cat, prev[~prev["table"].isin(cat["table"])]], ignore_index=True)
cat.to_csv(SILVER / "_catalog.csv", index=False)
print(cat.to_string(index=False))
print("silver total:", round(cat["size_mb"].sum(), 1), "MB")
spark.stop()

              table      rows  with_geometry  invalid_geometry  files  size_mb partitioned_by                                                                                                                                           source
  health_facilities    228986         228230                 0      1     23.7                                                                                                 FINAL COMPLETE DATASET/health/india_health_facilities_final.csv
      schools_india   1070017        1070016                 0     35    125.4     state_name                                                                                      FINAL COMPLETE DATASET/schools/india_schools_govt_aided.csv
     schools_kerala     11764          11763                 0      1      2.9                                                  FINAL COMPLETE DATASET/schools/kerala_schools_capacity.csv + kerala_schools_govt_aided_final.csv (level flags)
      fire_stations       129            129